# Predicción de Gestos LSM con Modelo Entrenado

Este notebook carga un modelo Keras (`.h5`), un pipeline de preprocesamiento `joblib` y un `LabelEncoder` `joblib` para predecir la clase de un nuevo archivo de video.

## 1. Importar Librerías y Definir Clases

**Importante:** Debemos redefinir exactamente las mismas clases `BaseEstimator` y `TransformerMixin` que se usaron durante el entrenamiento para que `joblib` pueda cargar el pipeline correctamente.

In [1]:
import os
import cv2
import mediapipe as mp
import numpy as np
import joblib
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.models import load_model
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from tqdm import tqdm
import tkinter as tk
from tkinter import filedialog

2025-10-26 19:14:37.537903: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-26 19:14:37.548046: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761527677.558694  116805 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761527677.562092  116805 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761527677.570888  116805 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
class FeatureExtractor:
    def __init__(self):
        self.feature_vector_size = 21 * 3 * 2
        self._hands = None

    @property
    def hands(self):
        if self._hands is None:
            print("Inicializando modelo MediaPipe Hands...")
            mp_hands = mp.solutions.hands
            self._hands = mp_hands.Hands(
                static_image_mode=False,
                max_num_hands=2,
                min_detection_confidence=0.5,
                min_tracking_confidence=0.5
            )
        return self._hands 

    def __getstate__(self):
        state = self.__dict__.copy()
        state['_hands'] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._hands = None 

    def extract_hand_keypoints(self, video_path):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"Error: No se pudo abrir el video {video_path}")
            return None

        video_keypoints = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = self.hands.process(image_rgb) 
            frame_keypoints = np.zeros(self.feature_vector_size)

            if results.multi_hand_landmarks:
                for i, hand_landmarks in enumerate(results.multi_hand_landmarks):
                    if i >= 2: continue
                    landmarks = np.array(
                        [[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]
                    ).flatten()
                    start_index = i * (21 * 3)
                    end_index = start_index + len(landmarks)
                    frame_keypoints[start_index:end_index] = landmarks
            video_keypoints.append(frame_keypoints)
        cap.release()
        return np.array(video_keypoints)

    def calculate_kinematic_features(self, keypoints_sequence):
        if keypoints_sequence.shape[0] < 2:
            return np.zeros_like(keypoints_sequence)
        velocities = np.diff(keypoints_sequence, axis=0)
        velocities = np.vstack([np.zeros(velocities.shape[1]), velocities])
        return velocities

    def process_video(self, video_path):
        keypoints = self.extract_hand_keypoints(video_path)
        if keypoints is None or keypoints.shape[0] == 0:
            return None
        velocities = self.calculate_kinematic_features(keypoints)
        combined_features = np.hstack([keypoints, velocities])
        return combined_features

class VideoFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.extractor = None 

    def _get_extractor(self):
        if self.extractor is None:
            self.extractor = FeatureExtractor()
        return self.extractor

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        all_sequences = []
        print("Paso 1/3: Extrayendo características del video...")
        extractor = self._get_extractor() 
        
        for video_path in tqdm(X, desc="Extrayendo secuencia"):
            # Usar la variable 'extractor' local
            features = extractor.process_video(video_path) 
            if features is not None and features.shape[0] > 0:
                all_sequences.append(features)
        return all_sequences

    def __getstate__(self):
        state = self.__dict__.copy()
        state['extractor'] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self.extractor = None 

class SequencePadder(BaseEstimator, TransformerMixin):
    def __init__(self, percentile=95, target_frames_=0): # target_frames_ se cargará desde el pipeline
        self.percentile = percentile
        self.target_frames_ = target_frames_

    def fit(self, X, y=None):
        if self.target_frames_ == 0:
            print("Advertencia: target_frames_ no se cargó, ajustando a los datos actuales.")
            sequence_lengths = [seq.shape[0] for seq in X]
            if sequence_lengths:
                self.target_frames_ = int(np.percentile(sequence_lengths, self.percentile))
        print(f"Longitud de secuencia objetivo: {self.target_frames_} fotogramas.")
        return self

    def transform(self, X, y=None):
        print("Paso 2/3: Estandarizando longitud de secuencia...")
        processed_sequences = []
        for seq in tqdm(X, desc="Padding"):
            num_frames, num_features = seq.shape
            if num_frames > self.target_frames_:
                processed_seq = seq[:self.target_frames_, :]
            elif num_frames < self.target_frames_:
                pad_width = self.target_frames_ - num_frames
                padding = np.zeros((pad_width, num_features))
                processed_seq = np.vstack([seq, padding])
            else:
                processed_seq = seq
            processed_sequences.append(processed_seq)
        return np.array(processed_sequences)

class TemporalFeatureProcessor(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=0.95, feature_pipeline_=None):
        self.n_components = n_components
        self.feature_pipeline_ = feature_pipeline_

    def fit(self, X, y=None):
        if self.feature_pipeline_ is None:
            print("Advertencia: feature_pipeline_ no se cargó, ajustando a los datos actuales.")
            self.feature_pipeline_ = Pipeline([\
                ('scaler', MinMaxScaler()),
                ('pca', PCA(n_components=self.n_components))
            ])
            n_videos, n_frames, n_features = X.shape
            reshaped_data = X.reshape(-1, n_features)
            self.feature_pipeline_.fit(reshaped_data)
        print("Scaler y PCA listos.")
        return self

    def transform(self, X, y=None):
        print("Paso 3/3: Aplicando Scaler y PCA...")
        n_videos, n_frames, n_features = X.shape
        reshaped_data = X.reshape(-1, n_features)
        transformed_data = self.feature_pipeline_.transform(reshaped_data)
        n_transformed_features = transformed_data.shape[1]
        final_data = transformed_data.reshape(n_videos, n_frames, n_transformed_features)
        print(f"Características reducidas a {n_transformed_features} componentes.")
        return final_data

print("Librerías y clases definidas.")

Librerías y clases definidas.


## 2. Indicar Rutas de Archivos

Introduce las rutas a tus archivos guardados. **Nota:** Estos archivos debes haberlos generado y guardado durante el entrenamiento.

In [3]:
root = tk.Tk()
root.withdraw()

notebook_dir = Path.cwd() 
parent_dir = notebook_dir
pipeline_path = parent_dir / "processed_data_pipeline/pipeline_procesamiento.joblib"
encoder_path = parent_dir / "processed_data_pipeline/label_encoder.joblib"

model_path = ""
print("Abriendo explorador para seleccionar el archivo del modelo (.h5)...")
model_path = filedialog.askopenfilename(
    title="Selecciona el archivo del modelo (.h5)",
    filetypes=[("Archivos HDF5", "*.h5"), ("Modelos Joblib", "*.joblib"), ("Todos los archivos", "*.*")]
)
print(f"Ruta de modelo seleccionada: {model_path}")

# print("\nAbriendo explorador para seleccionar el video (.mp4)...")
# video_path = filedialog.askopenfilename(
#     title="Ruta al video que quieres predecir (.mp4)",
#     filetypes=[("Archivos MP4", "*.mp4"), ("Archivos de Video", "*.avi;*.mov"), ("Todos los archivos", "*.*")]
# )
# print(f"Video seleccionado: {video_path}")

# print("\nAbriendo explorador para seleccionar la CARPETA de videos...")
# folder_path_str = filedialog.askdirectory(
#     title="Selecciona la carpeta con los videos a predecir"
# )

folder_path_str = "/mnt/c/Users/Raven/Desktop/ProyectoIntegrador/test"

video_files_list = []
if folder_path_str:
    folder_path = Path(folder_path_str)
    print(f"Carpeta seleccionada: {folder_path_str}")
    
    # Buscar videos con las extensiones deseadas
    video_extensions = ["*.mp4", "*.avi", "*.mov"]
    video_path_generators = [folder_path.glob(ext) for ext in video_extensions]
    
    # Combinar los generadores y convertir a lista de strings
    all_video_paths = [str(p) for gen in video_path_generators for p in gen]
    
    video_files_list = sorted(all_video_paths) # Ordenar alfabéticamente
    print(f"Se encontraron {len(video_files_list)} videos.")
else:
    print("No se seleccionó ninguna carpeta.")

root.destroy()


Abriendo explorador para seleccionar el archivo del modelo (.h5)...
Ruta de modelo seleccionada: /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/models/model_2_gru_2025-10-26_19-09-44.h5
Carpeta seleccionada: /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/test
Se encontraron 4 videos.


## 3. Cargar Artefactos (Modelo, Pipeline, Codificador)

In [4]:
print(f"Cargando modelo desde {model_path}...")

model = load_model(model_path)
print("Modelo Keras (.h5) cargado.")
print("\n--- Resumen del Modelo Keras ---")
model.summary()

print(f"Cargando modelo desde {model_path}...")
model = load_model(model_path)
print("Modelo cargado.")

print(f"Cargando pipeline desde {pipeline_path}...")
pipeline = joblib.load(pipeline_path)
print("Pipeline cargado.")

print(f"Cargando codificador desde {encoder_path}...")
encoder = joblib.load(encoder_path)
print("Codificador cargado.")

print("\n--- Resumen del Modelo ---")
model.summary()


Cargando modelo desde /mnt/c/Users/Raven/Desktop/ProyectoIntegrador/github/TC5035-PI-E51/Avance5/models/model_2_gru_2025-10-26_19-09-44.h5...


I0000 00:00:1761527684.715272  116805 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21458 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


Modelo Keras (.h5) cargado.

--- Resumen del Modelo Keras ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 150, 64)        │        13,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 150, 64)        │        13,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,206 (168.78 KB)

 Trainable params: 43,204 (168.77 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

## 4. Procesar Video y Realizar Predicción

In [5]:
print(f"\n--- Procesando {len(video_files_list)} Videos de la Carpeta ---")
    
# 1. Procesar TODOS los videos con el pipeline cargado
# El pipeline está diseñado para recibir una lista de rutas
processed_videos = pipeline.transform(video_files_list)

print(f"\nVideos procesados. Forma final (Videos, Frames, Features): {processed_videos.shape}")

# 2. Realizar la predicción en lote (batch)
print("Realizando predicciones...")
all_prediction_probs = model.predict(processed_videos)

print("\n--- ¡Predicciones Completas! ---")

# 3. Iterar e interpretar resultados para CADA video
for i, video_path_str in enumerate(video_files_list):
    video_name = Path(video_path_str).name
    prediction_probs = all_prediction_probs[i] # Obtener las prob. para este video
    
    predicted_index = np.argmax(prediction_probs) # Es un array 1D
    predicted_label = encoder.inverse_transform([predicted_index])[0]
    confidence = np.max(prediction_probs) * 100
    
    print(f"\n--- Video: {video_name} ---")
    print(f"  Clase Predicha: {predicted_label}")
    print(f"  Confianza: {confidence:.2f}%")
    
    print("  --- Probabilidades por Clase ---")
    for j, class_name in enumerate(encoder.classes_):
        print(f"    {class_name}: {prediction_probs[j]*100:.2f}%")
    print("-" * 35) # Separador


--- Procesando 4 Videos de la Carpeta ---
Paso 1/3: Extrayendo características del video...


Extrayendo secuencia:   0%|                                                                                    | 0/4 [00:00<?, ?it/s]libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

libEGL warning: MESA-LOADER: failed to open swrast: /usr/lib/dri/swrast_dri.so: cannot open shared object file: No such file or directory (search paths /usr/lib/x86_64-linux-gnu/dri:\$${ORIGIN}/dri:/usr/lib/dri, suffix _dri)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Inicializando modelo MediaPipe Hands...


W0000 00:00:1761527686.391887  117009 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761527686.409486  117009 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/raven/miniconda3/envs/lsm/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Extrayendo secuencia: 100%|████████████████████████████████████████████████████████████████████████████| 4/4 [00:10<00:00,  2.56s/it]


Paso 2/3: Estandarizando longitud de secuencia...


Padding: 100%|███████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 4641.00it/s]
I0000 00:00:1761527696.643675  116932 cuda_dnn.cc:529] Loaded cuDNN version 91002


Paso 3/3: Aplicando Scaler y PCA...
Características reducidas a 6 componentes.

Videos procesados. Forma final (Videos, Frames, Features): (4, 150, 6)
Realizando predicciones...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step

--- ¡Predicciones Completas! ---

--- Video: abrir.mp4 ---
  Clase Predicha: Tu
  Confianza: 52.04%
  --- Probabilidades por Clase ---
    Abrir: 46.04%
    Avión: 1.33%
    Bicicleta: 0.60%
    Tu: 52.04%
-----------------------------------

--- Video: avion.mp4 ---
  Clase Predicha: Tu
  Confianza: 51.99%
  --- Probabilidades por Clase ---
    Abrir: 46.08%
    Avión: 1.34%
    Bicicleta: 0.59%
    Tu: 51.99%
-----------------------------------

--- Video: bicicleta.mp4 ---
  Clase Predicha: Tu
  Confianza: 53.44%
  --- Probabilidades por Clase ---
    Abrir: 44.71%
    Avión: 1.27%
    Bicicleta: 0.58%
    Tu: 53.44%
-----------------------------------

--- Video: tu.mp4 ---
  Clase Predicha: Tu
  Confianza: 54.48%
  --- Probabilidades por Clase ---
    Abrir: 43.72%
 